# Phase 2 - Diagnosis head  —  Rotax_916_iSWhich of the 8 sensor channels is faulted, and with which of the 6 fault types(none/Bias/Drift/Spike/Stuck-At/Noise). One-of-N **per channel**.

In [ ]:
import os, sys, json, timesys.path.insert(0, os.path.abspath("."))import numpy as np, tensorflow as tffrom tensorflow import kerasimport tf_data_pipeline as P, model_architectures as A, train_common as CC.report_gpu()

In [ ]:
# ---- SET THE ENGINE HERE. Run this notebook once per engine. ----ENGINE = "Rotax_916_iS"steps = C.describe(ENGINE)train_ds, val_ds, test_ds = C.datasets(ENGINE)

### Class weighting`none` dominates so heavily that plain inverse-frequency weighting pushes the headto predict faults everywhere. Inverse-**sqrt** frequency, capped, computed on train only.

In [ ]:
xi  = A.build_encoder_input()enc = A.build_encoder(xi)det = A.build_detection_head(enc)dia = A.build_diagnosis_head(enc)model = keras.Model(xi, {"y_detection": det, "y_diagnosis": dia}, name="diagnosis")model.summary()

In [ ]:
cw = C.diagnosis_class_weights(ENGINE)np.save(os.path.join(C.MODELS_DIR, f"diag_class_weights_{C.ENGINE_KEY[ENGINE]}.npy"), cw)print("class weights:", cw.round(3))

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(1e-3, clipnorm=1.0),              loss={"y_detection": keras.losses.BinaryCrossentropy(),                    "y_diagnosis": C.weighted_diagnosis_loss(cw)},              loss_weights={"y_detection": 1.0, "y_diagnosis": 50.0},              metrics={"y_diagnosis": [C.DiagnosisMacroF1(), C.DiagnosisFaultRecall()]})

In [ ]:
PHASE = "phase2_diagnosis"PREV  = "phase1_detection"ckpt  = C.ckpt_path(ENGINE, PHASE)if PREV: C.warm_start(model, C.ckpt_path(ENGINE, PREV))hist = model.fit(    train_ds.map(C.split_labels(["y_detection","y_diagnosis"])),    validation_data=val_ds.map(C.split_labels(["y_detection","y_diagnosis"])),    steps_per_epoch=steps["train"],    validation_steps=steps["val"],    epochs=20,    callbacks=C.callbacks(ckpt, monitor="val_y_diagnosis_macro_f1", mode="max"),    verbose=1,)print("saved:", ckpt)

In [ ]:
res = model.evaluate(test_ds.map(C.split_labels(["y_detection","y_diagnosis"])), steps=steps["test"], verbose=1, return_dict=True)print(json.dumps({k: float(v) for k, v in res.items()}, indent=2))with open(os.path.join(C.MODELS_DIR, f"{PHASE}_{C.ENGINE_KEY[ENGINE]}_test.json"), "w") as f:    json.dump({k: float(v) for k, v in res.items()}, f, indent=2)